# Owen Coder — Expert Fine-Tune
Fine-tunes **Qwen2.5-Coder 3B** on Attestor-verified code.

1. **Upload** `training_data.jsonl` (use the file icon on the left)
2. **Runtime → Change runtime type → T4 GPU**
3. **Runtime → Run all**
4. Download the GGUF when done

In [ ]:
!pip install -q "unsloth[colab-new]" datasets trl

In [ ]:
import json, os

TRAINING_DATA = "training_data_merged.jsonl"
if not os.path.exists(TRAINING_DATA):
    TRAINING_DATA = "training_data.jsonl"
assert os.path.exists(TRAINING_DATA), (
    "Upload training_data_merged.jsonl (or training_data.jsonl) first! Use the file icon on the left sidebar.")

rows = []
with open(TRAINING_DATA, "r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
print(f"Loaded {len(rows)} training examples from {TRAINING_DATA}")

In [ ]:
# === EXPERT CONFIG ===
BASE_MODEL = "Qwen/Qwen2.5-Coder-3B-Instruct"
MAX_SEQ_LENGTH = 4096
LORA_R = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05
EPOCHS = 6
BATCH_SIZE = 2
GRAD_ACCUM = 8
LR = 1e-4
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj", "lm_head",
]

SYSTEM_PROMPT = (
    "You are Owen Coder, a code generation engine trained on "
    "Attestor-verified Python. You write clean, secure, deterministic code. "
    "You never use network access, shell execution, subprocess, eval, or "
    "exec. You handle edge cases, validate inputs at boundaries, and prefer "
    "the standard library. Every function you write passes static analysis."
)

CHAT_TEMPLATE = """<|im_start|>system
{system}<|im_end|>
<|im_start|>user
{instruction}<|im_end|>
<|im_start|>assistant
{output}<|im_end|>"""

def format_row(row):
    return CHAT_TEMPLATE.format(
        system=SYSTEM_PROMPT,
        instruction=row["instruction"],
        output=row["output"],
    )

In [ ]:
from unsloth import FastLanguageModel

print("Loading base model ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

print("Applying LoRA ...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

In [ ]:
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

dataset = Dataset.from_list([{"text": format_row(r)} for r in rows])
print(f"Dataset: {len(dataset)} examples")

sample_tokens = tokenizer(dataset[0]["text"], return_tensors="pt")
print(f"Sample token length: {sample_tokens['input_ids'].shape[1]}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=WARMUP_RATIO,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        weight_decay=WEIGHT_DECAY,
        fp16=True,
        bf16=False,
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="owen-coder-lora",
        save_strategy="epoch",
        report_to="none",
    ),
)

print("\n=== TRAINING STARTED ===")
stats = trainer.train()
print(f"\nTraining loss: {stats.training_loss:.4f}")
print(f"Runtime: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
print("Saving LoRA adapter ...")
model.save_pretrained("owen-coder-lora")
tokenizer.save_pretrained("owen-coder-lora")

print("Merging and exporting GGUF (q4_k_m) ...")
model.save_pretrained_gguf(
    "owen-coder-merged",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF export done!")

In [ ]:
# === TEST THE MODEL ===
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

test_prompt = CHAT_TEMPLATE.format(
    system=SYSTEM_PROMPT,
    instruction="Write a Python function: Validate an email address and return True if valid, False otherwise.",
    output="",
).rsplit("<|im_end|>", 1)[0]

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.3, top_p=0.9)
result = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("=== GENERATED CODE ===")
print(result)

In [ ]:
# === DOWNLOAD THE GGUF ===
import glob
from google.colab import files

gguf_files = glob.glob("owen-coder-merged/*.gguf")
for f in gguf_files:
    print(f"Downloading {f} ({os.path.getsize(f)/1024/1024:.0f} MB) ...")
    files.download(f)

print("\nDone! On your machine:")
print("  1. Put the .gguf in C:\\Users\\mange\\Owen 4.2\\Attestor 4.2\\training\\")
print("  2. cd \"C:\\Users\\mange\\Owen 4.2\\Attestor 4.2\\training\"")
print("  3. ollama create owen-coder -f Modelfile")
print("  4. set OWEN_MODEL=owen-coder")